# 03 · Bayesian optimisation of Suzuki–Miyaura conditions

Even a modest screen (8 ligands × 6 bases × 4 solvents × 4 temperatures = 768 reactions) is too big to run exhaustively by hand. **Bayesian optimisation (BO)** decides which experiment to run next, using everything measured so far:

```
          ┌───────────────────────────────────────────────┐
          ▼                                               │
  results so far ──► surrogate model ──► acquisition ──► run the suggested
  (yields)           (prediction +       function         reaction(s) in the lab
                      uncertainty)       (which reaction  │
                                         is most worth    │
                                         running next?)   │
          ▲                                               │
          └──────────────── add new yields ◄──────────────┘
```

- The **surrogate** is usually a Gaussian process (GP). It predicts the yield of every untested reaction **and how uncertain that prediction is**.
- The **acquisition function** trades off *exploitation* (run what looks best) against *exploration* (run what we know least about).

**Plan for this notebook:**
1. Set up a **virtual lab** from the Perera Suzuki dataset (notebook 02). Every reaction has been run, so we can "measure" any yield instantly and test strategies for free.
2. Write a BO loop **from scratch** (GP + expected improvement) to see how it works.
3. Use **[BayBE](https://emdgroup.github.io/baybe/)**, an open-source BO package built for chemistry, with chemical encodings of the reagents.
4. Use **transfer learning**: data from one substrate speeds up optimisation for the next. This is exactly the situation when moving through the para-bromobenzene series.
5. Set up the template for **your own campaign**, using the aryl bromide descriptors from notebook 01.

**Further reading:**
- Shields *et al.*, *Nature* **2021**, 590, 89: "EDBO", BO for reaction optimisation, benchmarked against chemists.
- Torres *et al.*, *J. Am. Chem. Soc.* **2022**, 144, 19999: EDBO+ (multi-objective, changing search spaces).
- Frazier, *A Tutorial on Bayesian Optimization*, arXiv:1807.02811 (**2018**): a clear introduction to GPs and acquisition functions.
- Rasmussen & Williams, *Gaussian Processes for Machine Learning*, MIT Press **2006** (free online), ch. 2.
- The [BayBE documentation](https://emdgroup.github.io/baybe/), especially the user guides on *Campaigns*, *Parameters* and *Transfer Learning*.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
RDLogger.DisableLog("rdApp.*")

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

from baybe import Campaign
from baybe.parameters import (CategoricalParameter, CustomDiscreteParameter,
                              NumericalDiscreteParameter, TaskParameter)
from baybe.searchspace import SearchSpace
from baybe.targets import NumericalTarget
from baybe.recommenders import RandomRecommender
from baybe.utils.random import set_random_seed

warnings.filterwarnings("ignore")      # GP fitting is chatty about convergence
DATA = Path("data"); DATA.mkdir(exist_ok=True)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

## 1. A virtual lab

We pick one coupling from the Perera dataset as our "lab": **quinoline-6-boronic acid pinacol ester + an indazole aryl bromide**, an aryl bromide / aryl-BPin Suzuki coupling. All 12 ligands × 8 bases × 4 solvents = 384 conditions were run, so `measure()` can return the real yield of any condition.

**You should try this for your data set too!**

The pair is chosen because it is *hard*. Most conditions give poor yields, and only a handful come close to the best. Random screening will struggle.

In [ ]:
XLSX = DATA / "suzuki_perera.xlsx"
if not XLSX.exists():
    import urllib.request
    urllib.request.urlretrieve("https://github.com/rxn4chemistry/rxn_yields/raw/master/data/Suzuki-Miyaura/"
                               "aap9112_Data_File_S1.xlsx", XLSX)

raw = pd.read_excel(XLSX)
suzuki = pd.DataFrame({
    "quinoline": raw["Reactant_1_Short_Hand"].str.split(", ").str[1],
    "partner":   raw["Reactant_2_Name"].str.split(", ").str[1],
    "ligand":    raw["Ligand_Short_Hand"].str.strip().fillna("None"),
    "base":      raw["Reagent_1_Short_Hand"].fillna("None"),
    "solvent":   raw["Solvent_1_Short_Hand"],
    "yield":     raw["Product_Yield_PCT_Area_UV"],
})

CONDITIONS = ["ligand", "base", "solvent"]
lab = (suzuki[(suzuki.quinoline == "6-BPin-Q") & (suzuki.partner == "Bromide")]
       [CONDITIONS + ["yield"]].reset_index(drop=True))
_lookup = lab.set_index(CONDITIONS)["yield"]

def measure(conditions: pd.DataFrame) -> np.ndarray:
    # 'Run' reactions in the virtual lab: look up their measured yields.
    return np.array([_lookup[tuple(row[c] for c in CONDITIONS)] for _, row in conditions.iterrows()])

best_possible = lab["yield"].max()
print(f"{len(lab)} possible conditions, best yield {best_possible:.1f}%, median {lab['yield'].median():.1f}%")
print(f"conditions within 90% of the best: {(lab['yield'] >= 0.9 * best_possible).sum()}")
lab.sort_values("yield", ascending=False).head(8)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(lab["yield"], bins=30)
ax.axvline(0.9 * best_possible, color="k", ls="--", lw=1, label="90% of best")
ax.set(xlabel="yield (%)", ylabel="number of conditions", title="The virtual lab: most conditions are poor")
ax.legend();

## 2. Bayesian optimisation from scratch

### The surrogate: a Gaussian process

A GP assumes that **similar inputs give similar outputs**. "Similar" is defined by a *kernel* $k(x, x')$. With a Matérn kernel, two conditions that share ligand and base are treated as more alike than two that share nothing. Given the measured points, the GP returns a **mean** $\mu(x)$ (best guess) and a **standard deviation** $\sigma(x)$ (uncertainty) for every untested condition. The uncertainty is small near measured points and large far away.

### The acquisition function: expected improvement (EI)

If $y^*$ is the best yield found so far, EI asks how much, on average, condition $x$ will *beat* $y^*$:

$$\mathrm{EI}(x) = (\mu - y^* - \xi)\,\Phi(z) + \sigma\,\phi(z), \qquad z = \frac{\mu - y^* - \xi}{\sigma}$$

($\Phi$, $\phi$: normal CDF and PDF.) EI is large when $\mu$ is high (exploitation) **or** when $\sigma$ is high (exploration). The small $\xi$ nudges towards exploration.

We encode conditions with one-hot vectors to start with.

In [ ]:
X_all = pd.get_dummies(lab[CONDITIONS]).to_numpy(float)
y_all = lab["yield"].to_numpy()

def fit_gp(X, y):
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(0.01)
    return GaussianProcessRegressor(kernel, normalize_y=True, n_restarts_optimizer=2, random_state=0).fit(X, y)

def expected_improvement(mu, sigma, best, xi=0.01):
    sigma = np.maximum(sigma, 1e-9)
    z = (mu - best - xi) / sigma
    return (mu - best - xi) * norm.cdf(z) + sigma * norm.pdf(z)

In [ ]:
# A snapshot: 10 random experiments done. What does the GP believe, and what would EI choose next?
rng = np.random.default_rng(1)
tested = list(rng.choice(len(lab), 10, replace=False))
gp = fit_gp(X_all[tested], y_all[tested] / 100)
mu, sd = gp.predict(X_all, return_std=True)
ei = expected_improvement(mu, sd, y_all[tested].max() / 100)
ei[tested] = 0

order = np.argsort(y_all)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(y_all[order], "k.", ms=3, label="true yield (unknown to the model)")
ax1.plot(100 * mu[order], color="C0", lw=1, label="GP mean")
ax1.fill_between(range(len(order)), 100 * (mu - 2 * sd)[order], 100 * (mu + 2 * sd)[order],
                 color="C0", alpha=0.2, label="GP ± 2σ")
pos = {i: k for k, i in enumerate(order)}
ax1.plot([pos[i] for i in tested], y_all[tested], "o", color="C1", label="measured so far")
nxt = int(np.argmax(ei))
ax1.plot(pos[nxt], y_all[nxt], "*", color="C3", ms=15, label="EI's choice for the next experiment")
ax1.set(ylabel="yield (%)"); ax1.legend(fontsize=8, loc="upper left")
ax2.plot(ei[order], color="C3"); ax2.set(ylabel="EI", xlabel="all 384 conditions, sorted by true yield")
fig.tight_layout()
print("EI suggests:", dict(lab.loc[nxt, CONDITIONS]), f"→ true yield {y_all[nxt]:.1f}%")

In [ ]:
def run_campaign(strategy, n_init=5, n_total=40, seed=0, xi=0.01):
    # Sequential campaign (1 experiment at a time). Returns the best yield found after each experiment.
    rng = np.random.default_rng(seed)
    tested = list(rng.choice(len(lab), n_init, replace=False))
    while len(tested) < n_total:
        if strategy == "random":
            nxt = rng.choice(np.setdiff1d(np.arange(len(lab)), tested))
        else:
            gp = fit_gp(X_all[tested], y_all[tested] / 100)
            mu, sd = gp.predict(X_all, return_std=True)
            acq = expected_improvement(mu, sd, y_all[tested].max() / 100, xi=xi)
            acq[tested] = -np.inf
            nxt = int(np.argmax(acq))
        tested.append(nxt)
    return np.maximum.accumulate(y_all[tested])


def plot_traces(traces, ax, title):
    # traces: {label: array (n_seeds, n_experiments)} -> median and interquartile band
    for label, t in traces.items():
        n = np.arange(1, t.shape[1] + 1)
        ax.plot(n, np.median(t, axis=0), label=label)
        ax.fill_between(n, *np.percentile(t, [25, 75], axis=0), alpha=0.2)
    ax.axhline(best_possible, color="k", ls="--", lw=1, label="best possible")
    ax.set(xlabel="number of experiments", ylabel="best yield found (%)", title=title)
    ax.legend(fontsize=8)


SEEDS = range(20)
traces = {s: np.array([run_campaign(s, seed=k) for k in SEEDS]) for s in ["random", "EI"]}
fig, ax = plt.subplots(figsize=(7, 4.5))
plot_traces(traces, ax, "From-scratch BO vs random screening (20 repeats)")

### Your turn (Part 2)

1. Replace EI with **upper confidence bound**, $\mathrm{UCB} = \mu + \kappa\sigma$. Try κ = 0 (pure exploitation), 2 and 10. What happens, and why?
2. How many experiments does BO need, on average, to find a condition within 90% of the best? What about random screening?
3. Change `n_init` (the number of random experiments before BO starts) to 2 and to 20. Is there a sweet spot?
4. Look at the GP's fitted kernel (`gp.kernel_`). What do the length scale and noise level tell you?

## 3. BayBE: BO for real experimental campaigns

Writing your own loop teaches the idea, but for real experiments you want a tested package. [BayBE](https://emdgroup.github.io/baybe/) adds:
- **batches**: suggest several reactions at once (e.g. one per heating block position);
- **chemical encodings**: describe reagents with descriptors instead of one-hot labels;
- **transfer learning**: reuse data from related reactions (Part 4);
- tidy handling of mixed parameters (categorical, numerical, ...).

### Encoding the reagents with descriptors

With one-hot encoding, every ligand is equally different from every other ligand. With descriptors, the GP can learn that, say, "bulky, electron-rich, mono-phosphines work", and carry that over to ligands it has not tested yet. Here are simple, physically motivated descriptors:
- **Ligands**: size, number of P atoms (mono- vs bidentate), sp³ fraction (alkyl vs aryl phosphine), aromatic rings, flexibility. All are computed from SMILES with RDKit.
- **Bases**: p*K*<sub>a</sub> of the conjugate acid (water) and the radius of the counter-cation.
- **Solvents**: dielectric constant, Reichardt's *E*<sub>T</sub>(30) polarity, protic or not.

In [ ]:
LIGAND_SMILES = {
    "P(tBu)3":     "CC(C)(C)P(C(C)(C)C)C(C)(C)C",
    "P(Ph)3":      "c1ccc(P(c2ccccc2)c2ccccc2)cc1",
    "AmPhos":      "CN(C)c1ccc(P(C(C)(C)C)C(C)(C)C)cc1",
    "P(Cy)3":      "C1CCC(P(C2CCCCC2)C2CCCCC2)CC1",
    "P(o-Tol)3":   "Cc1ccccc1P(c1ccccc1C)c1ccccc1C",
    "CataCXium A": "CCCCP(C12CC3CC(CC(C3)C1)C2)C12CC3CC(CC(C3)C1)C2",
    "SPhos":       "COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1",
    "dtbpf":       "CC(C)(C)P(C(C)(C)C)[C-]1C=CC=C1.CC(C)(C)P(C(C)(C)C)[C-]1C=CC=C1.[Fe+2]",
    "XPhos":       "CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2)c(C(C)C)c1",
    "dppf":        "c1ccc(P(c2ccccc2)[C-]2C=CC=C2)cc1.c1ccc(P(c2ccccc2)[C-]2C=CC=C2)cc1.[Fe+2]",
    "Xantphos":    "CC1(C)c2cccc(P(c3ccccc3)c3ccccc3)c2Oc2c(P(c3ccccc3)c3ccccc3)cccc21",
}

def ligand_descriptors(smiles):
    m = Chem.MolFromSmiles(smiles)
    return {"MW": Descriptors.MolWt(m),
            "n_P": sum(a.GetSymbol() == "P" for a in m.GetAtoms()),
            "frac_sp3": rdMolDescriptors.CalcFractionCSP3(m),
            "n_arom_rings": rdMolDescriptors.CalcNumAromaticRings(m),
            "n_rot_bonds": rdMolDescriptors.CalcNumRotatableBonds(m)}

LIGAND_DESC = pd.DataFrame({k: ligand_descriptors(v) for k, v in LIGAND_SMILES.items()}).T
LIGAND_DESC.loc["None"] = 0.0
LIGAND_DESC["no_ligand"] = (LIGAND_DESC.index == "None").astype(float)

# pKa of the conjugate acid in water; Shannon ionic radius of the cation (Å, 6-coordinate)
BASE_DESC = pd.DataFrame({
    #          pKaH   cation_radius  organic  no_base
    "NaOH":   [15.7,  1.02,          0,       0],
    "KOH":    [15.7,  1.38,          0,       0],
    "LiOtBu": [17.0,  0.76,          0,       0],
    "K3PO4":  [12.3,  1.38,          0,       0],
    "NaHCO3": [6.4,   1.02,          0,       0],
    "CsF":    [3.2,   1.67,          0,       0],
    "Et3N":   [10.75, 0.0,           1,       0],
    "None":   [0.0,   0.0,           0,       1],
}, index=["pKaH", "cation_radius", "organic", "no_base"]).T.astype(float)

SOLVENT_DESC = pd.DataFrame({
    #        dielectric  ET30   protic
    "MeCN": [37.5,       45.6,  0],
    "THF":  [7.6,        37.4,  0],
    "DMF":  [36.7,       43.2,  0],
    "MeOH": [32.7,       55.4,  1],
}, index=["dielectric", "ET30", "protic"]).T.astype(float)

LIGAND_DESC.round(2)

In [ ]:
objective = NumericalTarget(name="yield").to_objective()     # maximise yield

# decorrelate=False: by default BayBE drops descriptor columns that correlate with others. With small,
# hand-picked tables that can make two reagents identical (e.g. NaOH and KOH), so we keep every column.
def searchspace(encoding):
    if encoding == "one-hot":
        params = [CategoricalParameter(c, lab[c].unique(), encoding="OHE") for c in CONDITIONS]
    else:
        params = [CustomDiscreteParameter("ligand", LIGAND_DESC, decorrelate=False),
                  CustomDiscreteParameter("base", BASE_DESC, decorrelate=False),
                  CustomDiscreteParameter("solvent", SOLVENT_DESC, decorrelate=False)]
    return SearchSpace.from_product(params)

# One round of the loop, by hand:
campaign = Campaign(searchspace("descriptors"), objective)
batch = campaign.recommend(batch_size=3)          # no data yet -> a diverse starting set
batch["yield"] = measure(batch)                   # in real life: run these in the lab
campaign.add_measurements(batch)
print(batch)
campaign.recommend(batch_size=3)                  # the next suggestions, now informed by the data

In [ ]:
def simulate(make_campaign, lookup, n_batches=12, batch_size=3, seeds=range(10), initial_data=None):
    # Run repeated virtual campaigns. Returns an array (n_seeds, n_experiments) of best-yield-so-far.
    look = lookup.set_index(CONDITIONS)["yield"]
    traces = []
    for seed in seeds:
        set_random_seed(seed)
        campaign = make_campaign()
        if initial_data is not None:
            campaign.add_measurements(initial_data)
        found = []
        for _ in range(n_batches):
            batch = campaign.recommend(batch_size=batch_size)
            batch["yield"] = [look[tuple(r[c] for c in CONDITIONS)] for _, r in batch.iterrows()]
            campaign.add_measurements(batch)
            found.extend(batch["yield"])
        traces.append(np.maximum.accumulate(found))
    return np.array(traces)


strategies = {
    "random":          lambda: Campaign(searchspace("one-hot"), objective, recommender=RandomRecommender()),
    "BO, one-hot":     lambda: Campaign(searchspace("one-hot"), objective),
    "BO, descriptors": lambda: Campaign(searchspace("descriptors"), objective),
}
bb_traces = {name: simulate(make, lab) for name, make in strategies.items()}

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_traces(bb_traces, ax, "BayBE, batches of 3 (10 repeats)")

**Don't expect descriptors to win automatically.** If the descriptors don't capture what controls the yield, they can make the GP *worse* than plain one-hot labels: the GP is told that two ligands are similar when, chemically, they behave differently. Our ligand descriptors are crude, e.g. nothing about electron donation or steric bulk around P.

### Your turn (Part 3)

1. Improve the ligand descriptors: add Tolman cone angles or buried volumes (MORFEUS), or the P charge and HOMO from xTB (notebook 01 functions), or take them from the Kraken database (Gensch *et al.*, *J. Am. Chem. Soc.* **2022**, 144, 1205). Does BO improve?
2. Try a batch size of 1 vs 6 with the same total number of experiments. How much do you lose by running experiments in parallel?
3. Repeat the benchmark on an *easy* pair, e.g. `6-I-Q` + `Boronic Ester`. Is BO still worth it there?

## 4. Transfer learning: the next substrate is not a fresh start

In the project you will optimise conditions for several para-substituted bromobenzenes. The second substrate should not start from zero: conditions that worked for 4-bromoanisole are a good guess for 4-bromotoluene.

BayBE handles this with a **`TaskParameter`**. All data goes into one GP that learns *how similar* the substrates are, but recommendations are made only for the active substrate.

The virtual-lab analogue: optimise **6-bromoquinoline + boronic ester**, using 20 random earlier experiments each on the chloro-, iodo- and triflate-quinolines.

In [ ]:
SUBSTRATES = ["6-Cl-Q", "6-Br-Q", "6-I-Q", "6-OTf-Q"]
esters = (suzuki[(suzuki.partner == "Boronic Ester") & suzuki.quinoline.isin(SUBSTRATES)]
          .rename(columns={"quinoline": "substrate"})[["substrate"] + CONDITIONS + ["yield"]])
target_lab = esters[esters.substrate == "6-Br-Q"]
history = esters[esters.substrate != "6-Br-Q"].groupby("substrate").sample(20, random_state=0)
print(f"best possible for 6-Br-Q: {target_lab['yield'].max():.1f}%;  prior data: {len(history)} reactions")

def transfer_campaign():
    params = [CustomDiscreteParameter("ligand", LIGAND_DESC, decorrelate=False),
              CustomDiscreteParameter("base", BASE_DESC, decorrelate=False),
              CustomDiscreteParameter("solvent", SOLVENT_DESC, decorrelate=False),
              TaskParameter("substrate", SUBSTRATES, active_values=["6-Br-Q"])]
    return Campaign(SearchSpace.from_product(params), objective)

tl_traces = {
    "random":                       simulate(strategies["random"], target_lab, n_batches=8),
    "BO, no prior data":            simulate(strategies["BO, descriptors"], target_lab, n_batches=8),
    "BO + data from other halides": simulate(transfer_campaign, target_lab, n_batches=8, initial_data=history),
}
best_possible_tl = target_lab["yield"].max()
fig, ax = plt.subplots(figsize=(7, 4.5))
for label, t in tl_traces.items():
    n = np.arange(1, t.shape[1] + 1)
    ax.plot(n, np.median(t, axis=0), label=label)
    ax.fill_between(n, *np.percentile(t, [25, 75], axis=0), alpha=0.2)
ax.axhline(best_possible_tl, color="k", ls="--", lw=1, label="best possible")
ax.set(xlabel="experiments on 6-Br-quinoline", ylabel="best yield found (%)",
       title="Transfer learning from related substrates")
ax.legend(fontsize=8);

### Your turn (Part 4)

1. Use prior data from only **one** other substrate. Does it matter which one? (The chloride is the least similar: think oxidative addition.)
2. How much prior data is useful? Try 5, 20 and 100 reactions per substrate.
3. An alternative to `TaskParameter`: describe the substrate with **descriptors** (a `CustomDiscreteParameter` with `active_values`), as in Part 5 below. What is the advantage when you add a *new* substrate that has no data yet?

## 5. Your own campaign

This is the template for the real experiments. How it works:
- Every reaction you run goes into **`data/my_experiments.csv`**, one row per reaction, including failures (yield 0). Never delete rows.
- Each time you have new results, re-run the cells below. The campaign is rebuilt from the CSV, so the CSV is the single source of truth.
- The next batch of suggestions is printed and saved to `results/next_batch_<date>.csv`.

The **aryl bromide** enters through its descriptors from notebook 01 (`results/aryl_bromide_descriptors.csv`). Data from *all* bromides you have run inform the model, but suggestions are only made for the `TARGET_BROMIDE`. That is the transfer-learning idea from Part 4, using chemistry rather than labels to link the substrates.

> **Before the first real experiment**, discuss and agree with your supervisor: the list of ligands, bases, solvents and temperatures you can actually run, the catalyst loading, concentration and reaction time (kept fixed), and how yield is measured (e.g. ¹H NMR or GC with an internal standard).

> **Two substrate dimensions.** Your reaction matrix varies the boronic acid as well as the bromide, so the template needs a second `CustomDiscreteParameter` for it (and a matching column in `my_experiments.csv`). Note that Ar¹–Br + Ar²–B(OH)₂ and Ar²–Br + Ar¹–B(OH)₂ give the *same* product: think about whether the model should treat such pairs as related, and how you could use them to test it.

In [ ]:
# ======================= EDIT THIS: your search space ======================================
TARGET_BROMIDE = "4-bromoanisole"           # must be a row name in aryl_bromide_descriptors.csv
BROMIDE_DESCRIPTORS = ["dft_LUMO", "dft_IP", "dft_q_lowdin_ipso", "dft_dipole"]   # your choice from notebook 01

MY_LIGANDS = ["P(Ph)3", "P(Cy)3", "SPhos", "XPhos", "dppf"]           # keys of LIGAND_SMILES, or add new ones
MY_BASES = {                                # pKaH (water), cation radius (Å)
    "K2CO3":  [10.3, 1.38],
    "Cs2CO3": [10.3, 1.67],
    "K3PO4":  [12.3, 1.38],
    "NaOH":   [15.7, 1.02],
}
MY_SOLVENTS = ["1,4-dioxane/H2O 4:1", "THF/H2O 4:1", "toluene/EtOH/H2O 4:1:1", "EtOH/H2O 1:1"]
MY_TEMPERATURES = [40, 60, 80, 100]         # °C
BATCH_SIZE = 4                              # how many reactions you can run in parallel
# ==========================================================================================

bromide_table = pd.read_csv(RESULTS / "aryl_bromide_descriptors.csv", index_col="name")[BROMIDE_DESCRIPTORS]
my_ligand_desc = pd.DataFrame({k: ligand_descriptors(LIGAND_SMILES[k]) for k in MY_LIGANDS}).T
my_base_desc = pd.DataFrame(MY_BASES, index=["pKaH", "cation_radius"]).T

my_space = SearchSpace.from_product([
    CustomDiscreteParameter("aryl_bromide", bromide_table, active_values=[TARGET_BROMIDE], decorrelate=False),
    CustomDiscreteParameter("ligand", my_ligand_desc, decorrelate=False),
    CustomDiscreteParameter("base", my_base_desc, decorrelate=False),
    CategoricalParameter("solvent", MY_SOLVENTS, encoding="OHE"),   # mixtures: one-hot is simplest
    NumericalDiscreteParameter("temperature_C", MY_TEMPERATURES),
])
PARAMS = ["aryl_bromide", "ligand", "base", "solvent", "temperature_C"]
print(f"{len(my_space.discrete.exp_rep)} candidate reactions for {TARGET_BROMIDE} "
      f"({len(bromide_table)} aryl bromides can contribute data)")

In [ ]:
EXPERIMENTS = DATA / "my_experiments.csv"
if not EXPERIMENTS.exists():               # first run: create an empty lab record
    pd.DataFrame(columns=["date", "lab_book_ref"] + PARAMS + ["yield", "notes"]).to_csv(EXPERIMENTS, index=False)

experiments = pd.read_csv(EXPERIMENTS)
done = experiments.dropna(subset=["yield"])
print(f"{len(done)} completed reactions in {EXPERIMENTS} "
      f"({(done.aryl_bromide == TARGET_BROMIDE).sum()} on {TARGET_BROMIDE})")

my_campaign = Campaign(my_space, objective)
if len(done):
    my_campaign.add_measurements(done[PARAMS + ["yield"]])

next_batch = my_campaign.recommend(batch_size=BATCH_SIZE)
out = RESULTS / f"next_batch_{pd.Timestamp.today():%Y-%m-%d}.csv"
next_batch.to_csv(out, index=False)
print(f"saved to {out}")
next_batch

In [ ]:
# What does the model currently believe? Predicted yield ± uncertainty for the suggested batch.
# (Only meaningful once you have some data. Before that, the GP just returns its prior.)
if len(done) >= 3:
    stats = my_campaign.posterior_stats(next_batch)
    display(pd.concat([next_batch.reset_index(drop=True), stats.reset_index(drop=True)], axis=1).round(1))
else:
    print("Not enough data yet: run the first batch!")

### Practical advice for running a BO campaign

- **Fix everything you are not optimising** (scale, concentration, catalyst loading, time, stirring, work-up, analysis) and write it down.
- **Record failures** as yield 0, together with a note (e.g. "no conversion", "decomposition"). Leaving them out biases the model.
- **Measure the noise.** Run the same reaction 2–3 times early on. If replicates differ by ±10%, the model can't resolve differences smaller than that.
- **Don't override the suggestions silently.** If a suggestion is unsafe or impractical, change the search space instead (and note why). If you also want to run your own "chemist's choice" reactions, that's fine: add them to the CSV like any other result.
- **When to stop:** when several batches in a row no longer improve the best yield, or when the predicted improvement (EI) becomes negligible.
- **Version control:** keep `my_experiments.csv` in a git repository or a backed-up folder, and keep the lab-book reference for every row.

### Your turn (Part 5)

1. Before running anything, use the virtual-lab code to estimate how many experiments a search space of your size might need.
2. After your first two batches: look at `my_campaign.posterior_stats(...)` for all candidate reactions. Which conditions does the model think are most promising, and how uncertain is it?
3. Once you have data for two or more bromides, check whether the best conditions shift with σ<sub>p</sub>. Does that match what you would expect from oxidative addition?